# My final bulk analysis

I read only my frozen `bulk_v1` sample and audited responses. I make no API calls. I keep my **primary association regressions** separate from **tree prediction evaluation**: each regression uses all 1,000 outcomes under its prespecified formula, while each tree is fitted only on the 800 training cases and tested once on the 200 held-out cases. I do not tune features, depth, thresholds or prompts using test performance.

I report ordinary per-model confidence intervals and Holm-adjust the two primary rating p-values. I do not pool paired puzzles as independent model-comparison rows. I apply the class-count/convergence gates in my frozen analysis plan. Until a full collection is complete, I show progress only and do not preview held-out performance.

## 1. I verify my sample, protocol and response audits

In [1]:

from pathlib import Path
import json, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tools.sm_exceptions import PerfectSeparationWarning
from statsmodels.stats.multitest import multipletests
from scipy.stats import rankdata
import joblib, nbformat
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'experiments/stage_index.json').exists())
BASE = ROOT/'data/experiments/bulk_v1'
OUT = ROOT/'data/analysis/bulk_v1'; OUT.mkdir(exist_ok=True)
CFG = ROOT/'experiments/bulk_v1'
CONFIG = json.loads((CFG/'config.json').read_text())
PLAN = json.loads((CFG/'analysis_plan.json').read_text())
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def save(path, value): Path(path).write_text(json.dumps(value, indent=2)+'\n', encoding='utf-8')
def code_hash(name):
    n = nbformat.read(ROOT/'notebooks'/name, as_version=4)
    return hashlib.sha256('\n\n'.join(c.source for c in n.cells if c.cell_type=='code' and 'run_control' not in c.metadata.get('tags', [])).encode()).hexdigest()
manifest = json.loads((BASE/'manifest.json').read_text())
assert manifest['config_sha256'] == sha(CFG/'config.json') and manifest['plan_sha256'] == sha(CFG/'analysis_plan.json')
assert manifest['analysis_code_sha256'] == code_hash('06_final_analysis.ipynb')
assert manifest['collection_code_sha256'] == code_hash('04_bulk_collection.ipynb')
sampling = json.loads((BASE/'sampling.json').read_text())
assert sampling['sample_sha256'] == sha(BASE/'puzzles.csv')
puzzles = pd.read_csv(BASE/'puzzles.csv', dtype={'puzzle_id': str})
assert len(puzzles) == 1000 and puzzles.split.eq('train').sum()==800 and puzzles.split.eq('test').sum()==200
models = {}; progress = []
for name, spec in CONFIG['models'].items():
    folder = BASE/name; results = folder/'results.csv'
    if not results.exists(): progress.append({'Model':spec['label'],'Completed':0,'Planned':1000,'Analysis':'Pending'}); continue
    audit = json.loads((folder/'audit.json').read_text()); assert audit['passed']
    assert audit['sample_sha256'] == sampling['sample_sha256'] and audit['results_sha256'] == sha(results)
    assert audit['manifest_sha256'] == sha(BASE/'manifest.json')
    data = pd.read_csv(results, dtype={'puzzle_id':str})
    assert len(data)==1000 and data.puzzle_id.nunique()==1000 and set(data.puzzle_id)==set(puzzles.puzzle_id)
    assert data.set_index('puzzle_id')['split'].to_dict()==puzzles.set_index('puzzle_id')['split'].to_dict()
    completed = int(data.status.eq('completed').sum())
    progress.append({'Model':spec['label'],'Completed':completed,'Planned':1000,'Analysis':'Ready' if completed==1000 else 'Pending'})
    if completed == 1000:
        assert data.success.isin([0,1]).all()
        models[name]={'data':data,'label':spec['label'],'results_sha256':sha(results)}
display(pd.DataFrame(progress))
if not models: print('I keep the held-out evaluation closed until 1,000 audited responses are available for a model.')


,Model,Completed,Planned,Analysis
0,GPT-OSS 120B,0,1000,Pending
1,Gemini 3.5 Flash-Lite,404,1000,Pending


I keep the held-out evaluation closed until 1,000 audited responses are available for a model.


## 2. I fit the prespecified models and evaluate my held-out tree predictions

In [2]:
def fit_tree(frame,target,max_depth=2,min_leaf=10):
    values=frame.to_numpy(dtype=float);labels=np.asarray(target,dtype=int)
    assert len(values)==len(labels) and len(labels)>0 and np.isin(labels,[0,1]).all()
    assert np.isfinite(values).all()
    names=list(frame.columns)
    def grow(indices,depth):
        n=len(indices);positive=int(labels[indices].sum());p=positive/n
        node={'n':n,'correct':positive,'probability':p,'gini':2*p*(1-p),'predicted_class':int(p>=.5)}
        if depth>=max_depth or n<2*min_leaf or positive in [0,n]:return node
        best_gain=0.;best=None
        for j,name in enumerate(names):
            unique=np.unique(values[indices,j])
            for threshold in (unique[:-1]+unique[1:])/2:
                left=indices[values[indices,j]<=threshold];right=indices[values[indices,j]>threshold]
                if min(len(left),len(right))<min_leaf:continue
                lp=float(labels[left].mean());rp=float(labels[right].mean())
                gain=node['gini']-(len(left)*2*lp*(1-lp)+len(right)*2*rp*(1-rp))/n
                if gain>best_gain+1e-12:best_gain=gain;best=(name,float(threshold),left,right)
        if best is not None:
            name,threshold,left,right=best
            node.update(feature=name,threshold=threshold,gini_gain=best_gain,left=grow(left,depth+1),right=grow(right,depth+1))
        return node
    return grow(np.arange(len(labels)),0)


def tree_probabilities(tree, frame):
    result=[]
    for row in frame.to_dict('records'):
        node=tree
        while 'feature' in node:
            node=node['left'] if row[node['feature']]<=node['threshold'] else node['right']
        result.append(node['probability'])
    return np.asarray(result)

def wilson(correct,n):
    p=correct/n;z=1.96;center=(p+z*z/(2*n))/(1+z*z/n)
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/(1+z*z/n)
    return [float(center-half),float(center+half)]

features=['puzzle_rating','total_solver_moves','legal_moves']
metrics=[];rating_tests=[]
for name,item in models.items():
    data=item['data'];folder=OUT/name;folder.mkdir(exist_ok=True)
    positives=int(data.success.sum());negatives=1000-positives
    item['correct']=positives;item['accuracy']=positives/1000
    fit=None;tree=None;coeff=None
    if min(positives,negatives)>=50:
        x=pd.DataFrame({'const':np.ones(len(data)), 'rating_500':(data.puzzle_rating-1500)/500,
                        'multi_move':data.total_solver_moves.gt(1).astype(int)})
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            try:
                fit=sm.GLM(data.success.astype(int),x,family=sm.families.Binomial()).fit()
                assert fit.converged and np.isfinite(fit.params).all() and np.isfinite(fit.bse).all()
                assert not any(isinstance(w.message,PerfectSeparationWarning) for w in caught)
                ci=fit.conf_int()
                coeff=pd.DataFrame({'term':fit.params.index,'coefficient':fit.params.values,
                    'odds_ratio':np.exp(fit.params.values),'or_ci_low':np.exp(ci[0].values),
                    'or_ci_high':np.exp(ci[1].values),'p_value':fit.pvalues.values})
                assert np.isfinite(coeff.select_dtypes('number').to_numpy()).all()
            except (ValueError,AssertionError,np.linalg.LinAlgError): fit=None;coeff=None
        if fit is not None:
            coeff.to_csv(folder/'logistic_coefficients.csv',index=False)
            joblib.dump(fit,folder/'logistic_regression.joblib')
            rating_tests.append({'model':item['label'],'p_value':float(fit.pvalues['rating_500'])})
    train=data.loc[data.split.eq('train')].copy();test=data.loc[data.split.eq('test')].copy()
    assert len(train)==800 and len(test)==200
    if min(int(train.success.sum()),800-int(train.success.sum()))>=30:
        tree=fit_tree(train[features],train.success.astype(int),max_depth=2,min_leaf=40)
        # I freeze the fitted tree and training IDs before computing held-out metrics.
        tree_artifact={'tree':tree,'features':features,'train_ids':sorted(train.puzzle_id),
                       'test_ids':sorted(test.puzzle_id),'results_sha256':item['results_sha256'],
                       'analysis_plan_sha256':sha(CFG/'analysis_plan.json')}
        if (folder/'decision_tree.json').exists(): assert json.loads((folder/'decision_tree.json').read_text())==tree_artifact
        else: save(folder/'decision_tree.json',tree_artifact)
        probability=tree_probabilities(tree,test[features]);prediction=(probability>=.5).astype(int)
        truth=test.success.to_numpy(dtype=int);tp=int(((prediction==1)&(truth==1)).sum());tn=int(((prediction==0)&(truth==0)).sum())
        fp=int(((prediction==1)&(truth==0)).sum());fn=int(((prediction==0)&(truth==1)).sum())
        n1=int(truth.sum());n0=200-n1
        auc=float((rankdata(probability)[truth==1].sum()-n1*(n1+1)/2)/(n1*n0)) if n1 and n0 else None
        correct=tp+tn;baseline=int(train.success.mean()>=.5)
        metric={'model':item['label'],'train_n':800,'test_n':200,'tree_correct':correct,'tree_accuracy':correct/200,
                'accuracy_ci_95':wilson(correct,200),'balanced_accuracy':float((tp/n1+tn/n0)/2) if n1 and n0 else None,
                'roc_auc':auc,'brier_score':float(np.mean((probability-truth)**2)),
                'tp':tp,'tn':tn,'fp':fp,'fn':fn,'training_majority_class':baseline,
                'majority_baseline_test_accuracy':float(np.mean(truth==baseline))}
        save(folder/'heldout_evaluation.json',metric);metrics.append(metric)
        pd.DataFrame({'puzzle_id':test.puzzle_id,'truth':truth,'probability':probability,'prediction':prediction}).to_csv(folder/'heldout_predictions.csv',index=False)
    item.update(fit=fit,tree=tree,coefficients=coeff)
    save(folder/'status.json',{'n':1000,'correct':positives,'regression_fitted':fit is not None,'tree_fitted':tree is not None,
         'scope':'Separate fixed association model; predictive tree fitted on training data and evaluated on held-out puzzles.'})
if len(rating_tests)==2:
    adjusted=multipletests([r['p_value'] for r in rating_tests],method='holm')[1]
    for row,p in zip(rating_tests,adjusted): row['holm_adjusted_p_value']=float(p)
elif rating_tests:
    for row in rating_tests: row['holm_adjusted_p_value']=None
if rating_tests: pd.DataFrame(rating_tests).to_csv(OUT/'primary_rating_tests.csv',index=False)
if metrics:
    pd.DataFrame(metrics).to_csv(OUT/'tree_test_metrics.csv',index=False)
    display(pd.DataFrame(metrics)[['model','tree_accuracy','balanced_accuracy','roc_auc','majority_baseline_test_accuracy']])
save(OUT/'status.json',{'complete_models':list(models),'planned_per_model':1000,'tree_holdout_n':200,
                      'primary_rating_tests':rating_tests,'plan_sha256':sha(CFG/'analysis_plan.json')})


## 3. I save three final figures

In [3]:

# I create final figures only after both full collections are complete.
if len(models)==2:
    FIG=ROOT/'figures/bulk_results';FIG.mkdir(exist_ok=True)
    fig,ax=plt.subplots(figsize=(8,4))
    for i,item in enumerate(models.values()):
        p=item['accuracy'];low,high=wilson(item['correct'],1000)
        ax.bar(i,p,color=['#2563eb','#16a34a'][i],width=.55)
        ax.errorbar(i,p,yerr=[[p-low],[high-p]],fmt='none',color='black',capsize=4)
        ax.text(i,p+.1,f"{item['correct']}/1000",ha='center')
    ax.set(xticks=[0,1],xticklabels=[m['label'] for m in models.values()],ylim=(0,1),xlim=(-.65,1.65),
           ylabel='Exact first-move correctness',title='My balanced 1,000-puzzle study: descriptive accuracy')
    fig.tight_layout();fig.savefig(FIG/'accuracy.png',dpi=180);plt.show()
    fig,axes=plt.subplots(1,2,figsize=(11,4.4),sharex=True,sharey=True)
    for ax,item in zip(axes,models.values()):
        ax.set(title=item['label'],xlabel='Human puzzle rating',ylim=(-.05,1.05))
        if item['fit'] is None: ax.text(.5,.5,'Regression gate not met',ha='center',transform=ax.transAxes);continue
        rating=np.linspace(item['data'].puzzle_rating.min(),item['data'].puzzle_rating.max(),200)
        for group,color,label in [(0,'#2563eb','One solver move'),(1,'#dc2626','Multiple solver moves')]:
            grid=pd.DataFrame({'const':np.ones(200),'rating_500':(rating-1500)/500,'multi_move':np.full(200,group)})
            pred=item['fit'].get_prediction(grid).summary_frame();ax.plot(rating,pred['mean'],color=color,label=label)
            ax.fill_between(rating,pred['mean_ci_lower'],pred['mean_ci_upper'],color=color,alpha=.12)
    axes[0].set_ylabel('Predicted first-move success probability');axes[0].legend(fontsize=8)
    fig.suptitle('My separate logistic regressions; ordinary 95% model-based confidence bands',fontsize=11)
    fig.tight_layout();fig.savefig(FIG/'logistic_regression.png',dpi=180);plt.show()
    feature_labels={'puzzle_rating':'Rating','total_solver_moves':'Solver moves','legal_moves':'Legal moves'}
    def draw(ax,node,x=.5,width=.45,depth=0,parent=None,branch=None):
        y=1-depth*.35
        if parent:
            ax.plot([parent[0],x],[parent[1]-.04,y+.07],color='#64748b',zorder=0)
            ax.text((parent[0]+x)/2,(parent[1]+y)/2,branch,ha='center',fontsize=8)
        title=f"{feature_labels[node['feature']]} <= {node['threshold']:.1f}" if 'feature' in node else 'Leaf'
        ax.text(x,y,title+f"\ntrain n={node['n']}; correct={node['correct']}\nP={node['probability']:.2f}",ha='center',va='center',fontsize=8,
                bbox={'boxstyle':'round,pad=.35','facecolor':plt.cm.Blues(.15+.6*node['probability']),'edgecolor':'#475569'})
        if 'feature' in node:
            draw(ax,node['left'],x-width/2,width/2,depth+1,(x,y),'Yes');draw(ax,node['right'],x+width/2,width/2,depth+1,(x,y),'No')
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    for ax,item in zip(axes,models.values()):
        ax.set_axis_off();ax.set(xlim=(0,1),ylim=(.1,1.2),title=item['label'])
        if item['tree'] is None: ax.text(.5,.5,'Tree gate not met',ha='center');continue
        draw(ax,item['tree'])
    fig.suptitle('My trees fitted on 800 training puzzles; predictive performance is evaluated on 200 test puzzles',fontsize=11)
    fig.tight_layout();fig.savefig(FIG/'decision_trees.png',dpi=180);plt.show()
else: print('I keep my pilot figures unchanged while bulk collection is incomplete.')


I keep my pilot figures unchanged while bulk collection is incomplete.
